# SIS #3 — The Warehouse Sequence Detective
## RNN vs. LSTM: Does the order of events matter?

**CSCI1409 · Deep Learning · Bachelor’s degree · Week 11 · 4 points**
**Format:** In-class team assignment, followed by an individual oral defense
**Team size:** 1–3 students · **Submission:** one completed `.ipynb` per team on Microsoft Teams

### Scenario
A warehouse camera system detects these five actions:

| Event ID | Action |
|:--:|---|
| `0` | IDLE |
| `1` | CHECK |
| `2` | MOVE |
| `3` | SCAN |
| `4` | PACK |

Your task is to classify a sequence of events:

- **Label `1` (correct):** `SCAN` occurred **before** `PACK`.
- **Label `0` (incorrect):** `PACK` occurred **before** `SCAN`.

Every sequence contains **exactly one SCAN and one PACK**, plus unrelated actions. So knowing *which* actions occurred is not enough: your model must learn **their order**.

**Example:** `IDLE → SCAN → MOVE → PACK` is class **1**, whereas `IDLE → PACK → MOVE → SCAN` is class **0**.

### What is provided
The **first code cell below is the entire provided dataset** and the only provided code. Run it once without changing it. It creates the train, validation and test sets in memory, using only NumPy. **No downloads, external files, data cleaning, or GPU are needed.**

The input arrays are `X_train`, `X_val`, `X_test` and the labels are `y_train`, `y_val`, `y_test`. Each input is a sequence of **16 one-hot event vectors** (5 features per time step).

### Your task — write ALL code yourselves
You must implement **every model, training loop, metric, and experiment from scratch**, using the **blank code cells** provided after the dataset. You may use standard PyTorch building blocks such as `nn.Linear`, `nn.RNN`, `nn.LSTM`, `CrossEntropyLoss`, `Adam` and `DataLoader`. You do **not** need to implement LSTM gates manually.

- **A. Explore + pipeline:** Inspect the data, then write reusable training and evaluation functions.
- **B. Baseline:** Train an order-blind model.
- **C. RNN:** Build and train a vanilla RNN classifier.
- **D. LSTM:** Build and train an LSTM classifier.
- **E. Test + reversal:** Evaluate all models on the test set once, then reverse the test sequences and compare predictions.
- **F. Conclusions:** Report your results and answer the short questions.
- **G. Optional challenges** (not graded).

Every model trains in a few seconds on a laptop CPU. Spend your time on **predicting** results before you run, **reading** your metrics, and **explaining** why each model behaves as it does: that is what the questions and the defense are about.

### Reproducibility
- Define `SEED = 0`. Call `torch.manual_seed(SEED)` **immediately before creating each model**, and seed the mini-batch shuffling, e.g. `DataLoader(..., shuffle=True, generator=torch.Generator().manual_seed(SEED))`.
- The numbers in your Part F table must be the ones your notebook prints after **Kernel → Restart & Run All**.

### Rules and AI policy
This assignment follows the course’s **Level C (AI-Assisted)** policy.

- **Chat-based AI is allowed** for explanations, debugging and individual code snippets. You must **write, assemble, run, verify and understand** your own complete solution.
- **AI coding agents and code-editing assistants are not allowed:** no Codex, Claude Code, IDE-integrated coding agents, or tools that automatically create/edit/run your notebook for you. Use AI **only through a chat interface**.
- Cite any code you adapt from outside sources in a short comment. Verify everything an AI tells you: you will be asked to explain it.
- Do not change the provided dataset, labels or splits. Train on **train only**. In Parts A–D, check your models with the **validation** set only. The **test** set is used **once**, in Part E, after all models are trained.
- **All team members must participate and can be questioned on any part of the code**, not just the part they personally wrote.
- You are graded on correct implementation and understanding, **not** on achieving a particular accuracy threshold. Show your results even if a model performs poorly.

### Grading — 4.0 points

| Part | What you deliver | Points |
|---|---|---:|
| A | Data exploration; reusable training and evaluation functions | **0.50** |
| B | Order-blind baseline, trained and explained (F-Q1) | **0.50** |
| C | Vanilla RNN, trained and explained (F-Q2) | **0.75** |
| D | LSTM, trained and explained (F-Q3, F-Q4) | **0.75** |
| E | Test evaluation, reversal experiment and interpretation (F-Q5) | **0.50** |
| **Oral defense** | Individual understanding of the code and concepts | **1.00** |
| **Total** | | **4.00** |

Each part is graded on correct code, a reported result, and a correct explanation. Part G is optional and not graded.

**Suggested settings** (use the **same** settings for all models): `hidden_size=24`, `batch_size=128`, `epochs=10`, `learning_rate=0.01`, one recurrent layer. Use **2 output logits** for the two classes and cross-entropy loss.

---

### PROVIDED DATASET — run the following cell without editing it


In [32]:
import numpy as np

EVENTS = ["IDLE", "CHECK", "MOVE", "SCAN", "PACK"]
SEQUENCE_LENGTH = 16

def _make_split(n_samples, seed):
    rng = np.random.default_rng(seed)
    event_ids = rng.integers(0, 3, size=(n_samples, SEQUENCE_LENGTH), dtype=np.uint8)
    labels = np.array([0, 1] * (n_samples // 2), dtype=np.int64)
    rng.shuffle(labels)
    for i in range(n_samples):
        first, second = sorted(rng.choice(SEQUENCE_LENGTH, size=2, replace=False))
        event_ids[i, first], event_ids[i, second] = (
            (3, 4) if labels[i] == 1 else (4, 3)
        )
    return np.eye(5, dtype=np.float32)[event_ids], labels

X_train, y_train = _make_split(1200, seed=4)
X_val, y_val = _make_split(240, seed=5)
X_test, y_test = _make_split(240, seed=6)

print("Dataset ready — zero downloads required")
print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)


Dataset ready — zero downloads required
Train: (1200, 16, 5) (1200,)
Val:   (240, 16, 5) (240,)
Test:  (240, 16, 5) (240,)


## A — Explore the data and build the pipeline

### A1. Explore

**Write your own Python code** to:

1. Print the input shape and class counts for the train split.
2. Display **one example with label 1** and **one example with label 0** as readable event names, using the `EVENTS` list.
3. Inspect one time-step vector (for example, the first vector in the first sequence).

**Think about:** What do `N`, `T` and `5` represent in a shape `(N, T, 5)`? Why can two sequences containing the same number of SCAN and PACK events have different labels?

In [33]:
#1 
print(X_train.shape)
print(np.unique(y_train, return_counts=True))

#2
for i in range(len(y_train)):
    if y_train[i] == 1:
        sequence = []

        for step in X_train[i]:
            event_id = np.argmax(step)
            sequence.append(EVENTS[event_id])

        print("Label:", y_train[i])
        print("Sequence:", sequence)
        continue
    if y_train[i] == 0:
        sequence = []
        
        for step in X_train[i]:
            event_id = np.argmax(step)
            sequence.append(EVENTS[event_id])
        
        print("Label:", y_train[i])
        print("Sequence:", sequence)
        break
    
#3
print(X_train[0][0])


(1200, 16, 5)
(array([0, 1]), array([600, 600]))
Label: 1
Sequence: ['IDLE', 'IDLE', 'MOVE', 'MOVE', 'MOVE', 'IDLE', 'SCAN', 'MOVE', 'MOVE', 'IDLE', 'CHECK', 'MOVE', 'MOVE', 'PACK', 'MOVE', 'CHECK']
Label: 0
Sequence: ['MOVE', 'CHECK', 'MOVE', 'MOVE', 'CHECK', 'IDLE', 'MOVE', 'PACK', 'IDLE', 'MOVE', 'CHECK', 'MOVE', 'SCAN', 'MOVE', 'MOVE', 'IDLE']
[1. 0. 0. 0. 0.]


### A2. Reusable training and evaluation functions

Using **PyTorch**, implement your own code that:

- Converts the provided NumPy arrays to suitable PyTorch tensors (which dtype does `CrossEntropyLoss` expect for the labels?).
- Trains a model on the **training split** using mini-batches, cross-entropy loss and Adam, with seeded shuffling.
- Performs the normal training steps: forward pass, loss, clear old gradients, backward pass and optimizer update.
- Computes **accuracy** without updating the model: use `model.eval()` and `torch.no_grad()`.
- Prints the training loss and the **validation** accuracy after every epoch.

*You will reuse these functions in Parts B–E.*

In [34]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch import nn

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

def evaluate_accuracy(model, data_loader):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            logits = model(X_batch)
            predictions = torch.argmax(logits, dim=1)

            correct += (predictions == y_batch).sum().item()
            total += y_batch.size(0)

    return correct / total


# 4. Reusable training function
def train_model(
    model,
    train_dataset,
    val_dataset,
    epochs=10,
    batch_size=128,
    learning_rate=0.01,
    seed=0
):
    # Seeded shuffling
    generator = torch.Generator().manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=generator
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    for epoch in range(epochs):
        model.train()

        total_loss = 0.0
        total_samples = 0

        for X_batch, y_batch in train_loader:
            # Forward pass
            logits = model(X_batch)

            # Calculate loss
            loss = criterion(logits, y_batch)

            # Clear gradients from the previous batch
            optimizer.zero_grad()

            # Backward pass
            loss.backward()

            # Update model parameters
            optimizer.step()

            batch_size_actual = X_batch.size(0)
            total_loss += loss.item() * batch_size_actual
            total_samples += batch_size_actual

        average_loss = total_loss / total_samples
        val_accuracy = evaluate_accuracy(model, val_loader)

        print(
            f"Epoch {epoch + 1:2d}/{epochs} | "
            f"Train loss: {average_loss:.4f} | "
            f"Validation accuracy: {val_accuracy:.4f}"
        )

    return model

## B — Order-blind baseline

Create a small model that first **averages the 16 input vectors over the time dimension** and then applies a linear layer mapping **5 features → 2 class logits**.

Train it with your functions from Part A and report its **validation** accuracy. This model knows the **counts/proportions of events**, but not their order.

**Predict before you run:** can this model learn SCAN-before-PACK? Give a reason based on the *information lost by averaging*, then check your prediction.

In [35]:
# Prediction: averaging discards event positions, so this model should be near 50% accuracy.
class OrderBlindBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(5, 2)

    def forward(self, x):
        averaged = x.mean(dim=1)
        logits = self.linear(averaged)
        return logits
    
torch.manual_seed(0)

baseline_model = OrderBlindBaseline()

baseline_model = train_model(
    baseline_model,
    train_dataset,
    val_dataset,
    epochs=10,
    batch_size=128,
    learning_rate=0.01,
    seed=0
)
print(f"Baseline validation accuracy: {evaluate_accuracy(baseline_model, DataLoader(val_dataset, batch_size=128)):.4f}")


Epoch  1/10 | Train loss: 0.6959 | Validation accuracy: 0.4833
Epoch  2/10 | Train loss: 0.6928 | Validation accuracy: 0.4833
Epoch  3/10 | Train loss: 0.6928 | Validation accuracy: 0.4833
Epoch  4/10 | Train loss: 0.6931 | Validation accuracy: 0.4875
Epoch  5/10 | Train loss: 0.6929 | Validation accuracy: 0.5083
Epoch  6/10 | Train loss: 0.6928 | Validation accuracy: 0.5083
Epoch  7/10 | Train loss: 0.6924 | Validation accuracy: 0.4833
Epoch  8/10 | Train loss: 0.6926 | Validation accuracy: 0.4917
Epoch  9/10 | Train loss: 0.6923 | Validation accuracy: 0.4792
Epoch 10/10 | Train loss: 0.6925 | Validation accuracy: 0.4625
Baseline validation accuracy: 0.4625


## C — Vanilla RNN classifier

Implement a **new** PyTorch model using:

- A **single-layer `nn.RNN`** with `input_size=5`, `hidden_size=24`, `batch_first=True`.
- The **final hidden state** as the sequence representation.
- A linear classification head from **24 hidden features → 2 class logits**.

Train it with your functions from Part A and report its **validation** accuracy.

**Shape checkpoint:** With batch size `B`, the input is `(B, 16, 5)`; for a one-layer unidirectional RNN, the final hidden-state tensor is `(1, B, 24)`; the classifier output must be `(B, 2)`. **Print these three shapes for one small batch.**

**Defense checkpoint:** What information must the final hidden state carry for this task?

In [36]:
# Predict: unlike averaging, an RNN reads events in sequence and can remember
# whether SCAN or PACK came first. Its validation accuracy should exceed chance.
class VanillaRNNClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(input_size=5, hidden_size=24, batch_first=True)
        self.classifier = nn.Linear(24, 2)

    def forward(self, x):
        _, h_n = self.rnn(x)  # h_n: (1, batch, 24)
        return self.classifier(h_n[-1])  # (batch, 2)

torch.manual_seed(0)
rnn_model = VanillaRNNClassifier()

# Shape checkpoint before training.
small_batch = X_train_tensor[:8]
with torch.no_grad():
    _, hidden = rnn_model.rnn(small_batch)
    print("RNN shapes:", small_batch.shape, hidden.shape, rnn_model(small_batch).shape)

rnn_model = train_model(rnn_model, train_dataset, val_dataset)
rnn_val_accuracy = evaluate_accuracy(rnn_model, DataLoader(val_dataset, batch_size=128))
print(f"RNN validation accuracy: {rnn_val_accuracy:.4f}")


RNN shapes: torch.Size([8, 16, 5]) torch.Size([1, 8, 24]) torch.Size([8, 2])
Epoch  1/10 | Train loss: 0.6977 | Validation accuracy: 0.5792
Epoch  2/10 | Train loss: 0.6653 | Validation accuracy: 0.7292
Epoch  3/10 | Train loss: 0.5478 | Validation accuracy: 0.8583
Epoch  4/10 | Train loss: 0.2867 | Validation accuracy: 0.8958
Epoch  5/10 | Train loss: 0.1893 | Validation accuracy: 0.9375
Epoch  6/10 | Train loss: 0.0651 | Validation accuracy: 0.9667
Epoch  7/10 | Train loss: 0.1359 | Validation accuracy: 0.9667
Epoch  8/10 | Train loss: 0.0778 | Validation accuracy: 0.9750
Epoch  9/10 | Train loss: 0.0309 | Validation accuracy: 0.9583
Epoch 10/10 | Train loss: 0.0482 | Validation accuracy: 0.9875
RNN validation accuracy: 0.9875


## D — LSTM classifier

Implement a **new** PyTorch model using:

- A **single-layer `nn.LSTM`** with `input_size=5`, `hidden_size=24`, `batch_first=True`.
- The **final hidden state `h_n`**, not the cell state `c_n`, as the input to the classifier.
- A linear classification head from **24 hidden features → 2 class logits**.

Train it with **exactly the same settings as the RNN** and report its **validation** accuracy.

**API checkpoint:** The RNN returns `(outputs, h_n)`, while the LSTM returns `(outputs, (h_n, c_n))`.

**Defense checkpoint:** In simple words, what is the purpose of `c_n`? How is it different from `h_n`? **You do not need to implement the gates from scratch.**

In [37]:
# The LSTM cell state carries memory through the sequence; h_n is the final
# exposed hidden representation used for classification.
class LSTMClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=5, hidden_size=24, batch_first=True)
        self.classifier = nn.Linear(24, 2)

    def forward(self, x):
        _, (h_n, c_n) = self.lstm(x)
        return self.classifier(h_n[-1])  # c_n is not the classifier input

torch.manual_seed(0)
lstm_model = LSTMClassifier()
lstm_model = train_model(lstm_model, train_dataset, val_dataset)
lstm_val_accuracy = evaluate_accuracy(lstm_model, DataLoader(val_dataset, batch_size=128))
print(f"LSTM validation accuracy: {lstm_val_accuracy:.4f}")


Epoch  1/10 | Train loss: 0.6882 | Validation accuracy: 0.6542
Epoch  2/10 | Train loss: 0.6312 | Validation accuracy: 0.8333
Epoch  3/10 | Train loss: 0.3354 | Validation accuracy: 0.9167
Epoch  4/10 | Train loss: 0.1293 | Validation accuracy: 0.9333
Epoch  5/10 | Train loss: 0.0689 | Validation accuracy: 0.9833
Epoch  6/10 | Train loss: 0.0376 | Validation accuracy: 0.9917
Epoch  7/10 | Train loss: 0.0164 | Validation accuracy: 0.9917
Epoch  8/10 | Train loss: 0.0092 | Validation accuracy: 0.9958
Epoch  9/10 | Train loss: 0.0051 | Validation accuracy: 0.9958
Epoch 10/10 | Train loss: 0.0048 | Validation accuracy: 0.9875
LSTM validation accuracy: 0.9875


## E — Final test evaluation and the reversal experiment

All three models are now trained. Do not change any model or setting from here on.

1. Evaluate the baseline, RNN and LSTM on the **test set, once**. These are your reported test accuracies.
2. Reverse each test sequence **along the time dimension only** (keep its 5 features intact).
3. Check that the number of each type of event is unchanged by reversal.
4. Evaluate all three models on the reversed inputs **against the original `y_test` labels**. Label this as a **diagnostic**, not normal test accuracy.
5. Compare predictions: compute the **flip rate**, the fraction of test sequences whose predicted class changes after reversal.
6. Reason about the **correct** labels for the reversed sequences. (Hint: every sequence contains exactly one SCAN and one PACK.)

**Before you conclude anything:** a model can score close to 0% in step 4. Think about what that number means before you decide whether the model is broken.

**Questions:** Which models respond when the order changes? Why might the order-blind model give the same predictions after reversal?

In [38]:
# Freeze model choices here. The original test set is evaluated once per model.
models = {
    "Order-blind baseline": baseline_model,
    "Vanilla RNN": rnn_model,
    "LSTM": lstm_model,
}
val_loader = DataLoader(val_dataset, batch_size=128)
test_loader = DataLoader(test_dataset, batch_size=128)

# Flip only the time axis: (N, T, 5) stays (N, T, 5).
X_test_reversed = X_test_tensor.flip(dims=[1])
original_counts = X_test_tensor.sum(dim=1)
reversed_counts = X_test_reversed.sum(dim=1)
print("Event counts unchanged:", torch.equal(original_counts, reversed_counts))
print("Correct reversed labels:", (1 - y_test_tensor)[:8].tolist())

reversed_dataset = TensorDataset(X_test_reversed, y_test_tensor)
reversed_loader = DataLoader(reversed_dataset, batch_size=128)

def predict(model, inputs):
    model.eval()
    with torch.no_grad():
        return model(inputs).argmax(dim=1)

results = {}
for name, model in models.items():
    validation = evaluate_accuracy(model, val_loader)
    test = evaluate_accuracy(model, test_loader)
    reversed_diagnostic = evaluate_accuracy(model, reversed_loader)
    flip_rate = (predict(model, X_test_tensor) != predict(model, X_test_reversed)).float().mean().item()
    results[name] = (validation, test, reversed_diagnostic, flip_rate)
    print(f"{name}: val={validation:.4f}, test={test:.4f}, "
          f"reversed/original-labels={reversed_diagnostic:.4f}, flip={flip_rate:.4f}")

print("\n| Model | Validation | Test | Reversed diagnostic | Flip rate |")
print("|---|---:|---:|---:|---:|")
for name, values in results.items():
    print(f"| {name} | " + " | ".join(f"{v:.4f}" for v in values) + " |")


Event counts unchanged: True
Correct reversed labels: [1, 1, 1, 0, 0, 1, 1, 1]
Order-blind baseline: val=0.4625, test=0.5125, reversed/original-labels=0.5125, flip=0.0000
Vanilla RNN: val=0.9875, test=0.9917, reversed/original-labels=0.0125, flip=0.9792
LSTM: val=0.9875, test=1.0000, reversed/original-labels=0.0042, flip=0.9958

| Model | Validation | Test | Reversed diagnostic | Flip rate |
|---|---:|---:|---:|---:|
| Order-blind baseline | 0.4625 | 0.5125 | 0.5125 | 0.0000 |
| Vanilla RNN | 0.9875 | 0.9917 | 0.0125 | 0.9792 |
| LSTM | 0.9875 | 1.0000 | 0.0042 | 0.9958 |


## F — Results and conclusions

### Results table

Values from a fresh execution of Parts A–E with seed 0. The reversed column compares predictions with the *original* labels, so it is a diagnostic.

| Model | Validation accuracy | Test accuracy | Reversed vs. original labels (diagnostic) | Flip rate |
|---|---:|---:|---:|---:|
| Order-blind baseline | 0.4625 | 0.5125 | 0.5125 | 0.0000 |
| Vanilla RNN | 0.9875 | 0.9917 | 0.0125 | 0.9792 |
| LSTM | 0.9875 | 1.0000 | 0.0042 | 0.9958 |

### Short answers

1. Averaging removes the positions of SCAN and PACK. Two sequences with the same event counts then produce the same five averages even when their labels differ.

2. `B` is the number of sequences in a batch, and `T` is the number of time steps in each sequence (16 here). For `B=8`, the output logits have shape `(8, 2)`: two class scores for each of eight sequences.

3. The LSTM cell state `c_n` carries longer-term information through the sequence. The hidden state `h_n` is the exposed representation at the final time step, which the classifier uses.

4. The LSTM had the highest test score (1.0000 versus 0.9917), while both models had the same validation score (0.9875). The small difference on this single split does not establish a meaningful advantage, and it does not imply that LSTMs always outperform RNNs.

5. A near-zero reversed diagnostic is expected if the model reacts to order: reversal changes the correct label to `1 - y_test`, while this diagnostic keeps the old label. The RNN and LSTM flip rates are high, showing that their predictions usually change; the baseline flip rate is zero because its averages are unchanged.


## G — Optional challenges (not graded)

Only if you have finished A–F. None of these affects your 4 points, but they make good defense material.

- **G1. Is the difference real?** Retrain the RNN and LSTM with seeds `0, 1, 2` and compare their **validation** accuracies. Is the RNN–LSTM gap larger than the difference between two seeds of the same model?
- **G2. Confusion matrices.** Write your own 2×2 confusion matrix (rows = true class, columns = predicted class) and print it for each model on the original and the reversed test inputs. What pattern appears for the RNN after reversal, and why does the baseline’s matrix not change?
- **G3. Parameters.** Derive by hand the number of parameters of your RNN and LSTM, check with `sum(p.numel() for p in model.parameters())`, and explain why the LSTM has about 4× more.
- **G4. Longer sequences (no code).** In Homework 7 you computed how an early input’s influence shrinks as $w^{T-1}$. Using this, explain why a vanilla RNN would struggle if SCAN and PACK could be 50 steps before the end. How does the LSTM update $c_t = f \cdot c_{t-1} + i \cdot g$ avoid this when the forget gate $f$ is close to 1?

*Use the empty cell below if you attempt G1–G3, and write G4 as text.*

### Team members and contributions

- **Student 1** — Full name / student ID / contribution: …
- **Student 2** — Full name / student ID / contribution: …
- **Student 3** — Full name / student ID / contribution: …

### Before submitting

- **Kernel → Restart & Run All** finishes without errors, and the numbers in your Part F table match the printed outputs.
- The test set is evaluated only in Part E.
- Each team member can explain any chosen code line during the **individual oral defense**.

**Submission:** One completed `.ipynb` per team via Microsoft Teams. No separate report required.